In [15]:
import pandas as pd

df = pd.read_excel("Arabic_Depression_10.000_Tweets.xlsx")

print("Shape (rows, columns):", df.shape)
print("Columns:", df.columns.tolist())
print("\nLabel counts:\n", df["label"].value_counts())
df.head(10)

Shape (rows, columns): (10000, 3)
Columns: ['Unnamed: 0', 'tweet', 'label']

Label counts:
 label
1    5000
0    5000
Name: count, dtype: int64


,Unnamed: 0,tweet,label
0,0,والله شايفيتلك الناس كلها صاير عندها اكتئاب لا...,1
1,1,اكيد اكتئاب بعد العصر ما في زيه بس ان شاء الله...,1
2,2,والله ما أنا عارف هو اكتئاب ولا انطفاء ولا نضج...,1
3,3,اصبروا داخله في حاله اكتئاب 💔😭,1
4,4,هذا اكتئاب ولا,1
5,5,حزينه ل ابعد درجه اول مسلسل اصيح على هالكثر وم...,1
6,6,أنا لو اتعرضت علي دكتور نفسي هجبله اكتئاب بجد ...,1
7,7,ماعدت أعرف هو اكتئاب والا انطفاء ولا نضج,1
8,8,هذا اول اكتئاب جاني🥲💔,1
9,9,جاني اكتئاب اول ما شفته😔,1


In [16]:
# How many tweets in each class contain the word "depression"?
df["has_keyword"] = df["tweet"].astype(str).str.contains("اكتئاب")
print(pd.crosstab(df["label"], df["has_keyword"]))

# How long are the tweets (in words)?
df["n_words"] = df["tweet"].astype(str).str.split().str.len()
print("\nWord count stats:\n", df["n_words"].describe())

#few label 0 examples
df[df["label"] == 0].head(10)

has_keyword  False  True 
label                    
0             4998      2
1             3205   1795

Word count stats:
 count    10000.000000
mean        10.860200
std          6.770468
min          1.000000
25%          5.000000
50%          9.000000
75%         15.000000
max         59.000000
Name: n_words, dtype: float64


,Unnamed: 0,tweet,label,has_keyword,n_words
5000,0,لا تبخلوا بإسعاد من تحبون فكل المواقف تتلاشى س...,0,False,16
5001,1,الله يسعدكم سعادة الدارين,0,False,4
5002,2,الأرواح الجميلة تجبرك على أن تقترب منها و تستع...,0,False,20
5003,3,كل سنة وانت طيبة ي مروة ويارب سنينك الجايه ف...,0,False,14
5004,4,سلامتك ياروحي ماتشوفين شر يارب وجودك سعاده و...,0,False,11
5005,5,غباء منك أن تكون حزين بسبب شخص يعيش حياته بكل ...,0,False,11
5006,6,اللهم ارزقني فرحا لم أتوقع حدوثه يسع لهذه الدن...,0,False,18
5007,7,بدعيلك مشعل ومن كل قلبي ياكثر تدعي لنا وماتنس...,0,False,12
5008,8,اللهم بلغنا المتسع واليُسر في جميع أمورنا وتحق...,0,False,22
5009,9,الله يسعدك ياغيث انت وفريق العمل ومن اروع ...,0,False,15


In [17]:
import pandas as pd
import re             # re = "regular expressions", used to find and replace text patterns
import html           # html = fixes web codes like "&amp;" back into "&"

log = []   # empty list; each step adds one line to it , helps to keep track of how many rows we have after each step.


def log_step(name, data):
    log.append({
        "step": name,
        "rows": len(data),
        "label_1": int((data["label"] == 1).sum()), # how many are "depressed"
        "label_0": int((data["label"] == 0).sum()), # how many are "not depressed"
    })

df = pd.read_excel("Arabic_Depression_10.000_Tweets.xlsx") #loading again so we can start afresh

# old row number column. We don't need it.
df = df.drop(columns=["Unnamed: 0"])

log_step("0. raw", df)   # record the starting count (should be 10,000)


#***********************************************************************************************************
# 1. Remove empty tweets

df = df.dropna(subset=["tweet"])         # remove rows where the tweet cell is completely empty
df["tweet"] = df["tweet"].astype(str)    # make sure every tweet is treated as text
df = df[df["tweet"].str.strip() != ""]   # removing tweets that are only spaces

log_step("1. drop empty", df)


#*************************************************************************************************************
# 2. Lightly clean the text of each tweet

def clean_text(t):

    t = t.replace("_x000D_", " ")

    t = html.unescape(t)

    # Remove invisible characters
    t = re.sub(r"[\u200e\u200f\u202a-\u202e\u2066-\u2069]", "", t)

    # Remove "RT " at the start (means "retweet", not the person's own words)
    t = re.sub(r"^RT\s+", "", t)

    # Remove anys links (anything starting with http://, https:// or www.)
    t = re.sub(r"https?://\S+|www\.\S+", "", t)

    # Replace usernames like @ahmed with [USER] to protect privacy
    t = re.sub(r"@\w+", "[USER]", t)

    # Replace multiple spaces or line breaks with one single space, and remove spaces at the start and end.

    t = re.sub(r"\s+", " ", t).strip()

    return t

# Run clean_text() on every tweet in the table
df["tweet"] = df["tweet"].apply(clean_text)

# If a tweet was ONLY a link or username, it's now empty, so remove it
df = df[df["tweet"] != ""]

log_step("2. clean text", df)


#*************************************************************************************************************
# 3. Remove contradictory tweets:  If the exact same tweet is labelled 1 in one row and 0 in another, we can't trust either label, so we remove both.

label_counts = df.groupby("tweet")["label"].nunique()  # for each tweet text: how many different labels
conflicting = label_counts[label_counts > 1].index     # tweets with MORE than 1 label = contradictory
df = df[~df["tweet"].isin(conflicting)]                # keep only tweets NOT in that list

log_step("3. drop conflicting", df)


#*************************************************************************************************************
# 4. Remove exact duplicates (same tweet text appearing twice)- Keeps the first copy and removes the rest.

df = df.drop_duplicates(subset="tweet")

log_step("4. drop duplicates", df)


#------------------------------------------------------------------------------------------------------

#checking the log
log_df = pd.DataFrame(log) # turn the list into a table
print(log_df)

#-----------------------------------------------------------------------------------------------------


# Count words in each tweet (split on spaces, then count the pieces)
df["n_words"] = df["tweet"].str.split().str.len()

# Show length statistics separately for label 0 and label 1
print("\nWords per tweet, by label:")
print(df.groupby("label")["n_words"].describe())

# We need at least 50 of each label.
print("\nHow many tweets per label would survive a minimum length:")
for m in [5, 10, 15, 20]:
    kept = df[df["n_words"] >= m]["label"].value_counts()
    print(f"  >= {m} words ->  label 1: {kept.get(1,0)},  label 0: {kept.get(0,0)}")


                  step   rows  label_1  label_0
0               0. raw  10000     5000     5000
1        1. drop empty  10000     5000     5000
2        2. clean text  10000     5000     5000
3  3. drop conflicting  10000     5000     5000
4   4. drop duplicates   9962     4992     4970

Words per tweet, by label:
        count       mean       std  min  25%  50%   75%   max
label                                                        
0      4970.0  10.630382  6.126131  1.0  6.0  9.0  15.0  43.0
1      4992.0  10.926883  7.175845  1.0  5.0  9.0  16.0  59.0

How many tweets per label would survive a minimum length:
  >= 5 words ->  label 1: 4006,  label 0: 4147
  >= 10 words ->  label 1: 2375,  label 0: 2438
  >= 15 words ->  label 1: 1425,  label 0: 1305
  >= 20 words ->  label 1: 784,  label 0: 573


- This table shows that label 0 and label 1 have almost the same lengths.
- That means filtering by length won't unfairly remove more of one class than the other:
- We wanted the highest cutoff (so the pieces are as meaningful as possible) that still leaves at least 50 per label.

- All of them leave plenty, so I chose 15. It gives about 5 words per third, which is enough for a real phrase, while keeping over 1,300 per class.

In [18]:
# Keep long tweets, pick a balanced sample, save

MIN_WORDS = 15      # minimum tweet length (in words)
N_PER_CLASS = 50    # how many tweets we want from each label
SEED = 42           # fixed "random seed" = the random pick is:

#---------------------------------------------------------
# 1. Keep only tweets with at least MIN_WORDS words
df_long = df[df["n_words"] >= MIN_WORDS].copy()
log_step(f"5. keep >= {MIN_WORDS} words", df_long)

#----------------------------------------------------------
# 2. Add a helpful column: does the tweet contain the word "depression"?
#    We aren't removing these tweet, just marking them.

df_long["has_keyword"] = df_long["tweet"].str.contains("اكتئاب")


# 3. Take a random balanced sample:
# 50 tweets with label 1 + 50 with label 0
# --------------------------------------------------------
sample = (
    df_long.groupby("label", group_keys=False)    # split the table by label
           .apply(lambda g: g.sample(n=N_PER_CLASS, random_state=SEED))
)

# Shuffle the 100 rows so the 1s and 0s are mixed (not 50 ones, then 50 zeros)
sample = sample.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Give each post a simple ID like "ar_001", "ar_002"
sample.insert(0, "id", [f"ar_{i+1:03d}" for i in range(len(sample))])
sample.insert(1, "language", "Arabic")

log_step("6. balanced sample", sample)

#---------------------------------------------------------------------

#4.  The full cleaning log, start to finish
print(pd.DataFrame(log))
print("\nLabel counts in sample:\n", sample["label"].value_counts())
print("\nKeyword 'اكتئاب' in sample (by label):")
print(pd.crosstab(sample["label"], sample["has_keyword"]))
sample.head(10)

# ------------------------------------------------------------
# 5. Save the files
# all cleaned tweets >= 15 words

df_long.to_csv("arabic_clean_full.csv", index=False, encoding="utf-8-sig")
sample.to_csv("arabic_sample_100.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(log).to_csv("arabic_cleaning_log.csv", index=False)
print("\nSaved 3 files.")





                  step   rows  label_1  label_0
0               0. raw  10000     5000     5000
1        1. drop empty  10000     5000     5000
2        2. clean text  10000     5000     5000
3  3. drop conflicting  10000     5000     5000
4   4. drop duplicates   9962     4992     4970
5  5. keep >= 15 words   2730     1425     1305
6   6. balanced sample    100       50       50

Label counts in sample:
 label
1    50
0    50
Name: count, dtype: int64

Keyword 'اكتئاب' in sample (by label):
has_keyword  False  True 
label                    
0               50      0
1               33     17

Saved 3 files.


/tmp/ipykernel_3300/1790576714.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=N_PER_CLASS, random_state=SEED))


In [19]:
# STEP 4: Split each tweet into 3 equal parts - (prefix = start, middle, suffix = end)


def split_into_thirds(text):
    words = text.split()
    n = len(words)

    # Size of each part. If the words don't divide evenly by 3, the leftover 1 or 2 words go to the first part(s). Example: 16 words -> sizes 6, 5, 5
    base = n // 3
    extra = n % 3
    sizes = [base + (1 if i < extra else 0) for i in range(3)]

    prefix = words[:sizes[0]]                          # first chunk
    middle = words[sizes[0]:sizes[0] + sizes[1]]       # second chunk
    suffix = words[sizes[0] + sizes[1]:]               # everything left

    # Glue each chunk back into a sentence
    return " ".join(prefix), " ".join(middle), " ".join(suffix)


# Apply the function to every tweet in the sample, making 3 new columns
sample[["prefix", "middle", "suffix"]] = sample["tweet"].apply(
    lambda t: pd.Series(split_into_thirds(t))
)

# Rename "tweet" to "full" so the 4 versions are clearly named
sample = sample.rename(columns={"tweet": "full"})

# ------------------------------------------------------------
# Safety check: gluing the 3 parts back together should give
# exactly the full tweet (no words lost or duplicated)

rebuilt = sample["prefix"] + " " + sample["middle"] + " " + sample["suffix"]
print("All splits correct:", (rebuilt == sample["full"]).all())

kw = "اكتئاب"
sample["kw_in_prefix"] = sample["prefix"].str.contains(kw)
sample["kw_in_middle"] = sample["middle"].str.contains(kw)
sample["kw_in_suffix"] = sample["suffix"].str.contains(kw)
print("\nWhere the keyword appears (label 1 tweets only):")
print(sample[sample["label"] == 1][["kw_in_prefix", "kw_in_middle", "kw_in_suffix"]].sum())


cols = ["id", "language", "label", "full", "prefix", "middle", "suffix",
        "n_words", "has_keyword", "kw_in_prefix", "kw_in_middle", "kw_in_suffix"]

sample = sample[cols]
sample.to_csv("arabic_sample_split.csv", index=False, encoding="utf-8-sig")
print("\nSaved arabic_sample_split.csv")
sample.head(5)

All splits correct: True

Where the keyword appears (label 1 tweets only):
kw_in_prefix    4
kw_in_middle    7
kw_in_suffix    7
dtype: int64

Saved arabic_sample_split.csv


,id,language,label,full,prefix,middle,suffix,n_words,has_keyword,kw_in_prefix,kw_in_middle,kw_in_suffix
0,ar_001,Arabic,1,حزينه وفاقده كل الحنان والحزن في حياتي مطلقه ل...,حزينه وفاقده كل الحنان والحزن في حياتي,مطلقه له العنان وكل من شافني أشر,بالبنان ويقول هالبنت مليانه كآبه بالاطنان,20,False,False,False,False
1,ar_002,Arabic,1,انا كنت راجع من بره مبسوط فشخ و متحمس إني اتكل...,انا كنت راجع من بره مبسوط فشخ و متحمس,إني اتكلم و بتاع دخلت بيتنا عملولي حاله اكتئاب,فشيخه واله و انا خلاص امنيه حياتي اموت,26,True,False,True,False
2,ar_003,Arabic,1,لأسف العيادات النفسية ومن تجربة الدكتور لايقدر...,لأسف العيادات النفسية ومن تجربة الدكتور,لايقدر المريض اذكر في احدا,المرات ذهب لمستشفى الحبيب بريدة,16,False,False,False,False
3,ar_004,Arabic,0,الوفاء هو اعظم ما تتحلى به النفس البشريه . هو ...,الوفاء هو اعظم ما تتحلى به النفس البشريه,. هو ذاك الطبع الذي يعتبره البعض,نادرا في زمننا المليء بالجمود والنكران .,22,False,False,False,False
4,ar_005,Arabic,0,مليتياو من اول ما اشتراه مدريد و انا متفائل في...,مليتياو من اول ما اشتراه مدريد و,انا متفائل فيه بس انه انظلم,بالدكه واصابه راموس كانت نعمه والله,19,False,False,False,False


In [20]:
# (for the length histogram, so short tweets are included) - Selena this should help
df.to_csv("arabic_clean_before_length_filter.csv", index=False, encoding="utf-8-sig")
print("Saved:", len(df), "tweets")   # should print 9962

Saved: 9962 tweets
